## AUDITORIA DA CAMADA SILVER


In [2]:
from pathlib import Path
import sys

RAIZ = Path.cwd()

if not (RAIZ / "src" / "enem_pipeline" / "config.py").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "src" / "enem_pipeline" / "config.py").exists(), (
    "Não foi possível localizar src/enem_pipeline/config.py"
)

sys.path.insert(0, str(RAIZ / "src"))


import duckdb
import pandas as pd

from enem_pipeline.config import SILVER_DIR
from enem_pipeline.silver import obter_coluna_territorial


# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

UF = "RR"
DIRETORIO_SILVER = SILVER_DIR / f"uf={UF}"

ANOS_ESPERADOS = set(range(1998, 2026))

COLUNAS_IDENTIFICADORAS = [
    "NU_INSCRICAO",
    "NU_ANO",
]


# ============================================================
# 2. LOCALIZAR OS ARQUIVOS
# ============================================================

arquivos = sorted(DIRETORIO_SILVER.glob("ano=*/*.parquet"))

print("=" * 70)
print("AUDITORIA DA CAMADA SILVER")
print("=" * 70)
print(f"UF: {UF}")
print(f"Diretório: {DIRETORIO_SILVER}")
print(f"Arquivos encontrados: {len(arquivos)}")


# ============================================================
# 3. AUDITORIA INDIVIDUAL DOS ARQUIVOS
# ============================================================

resultados = []
con = duckdb.connect()

try:
    for arquivo in arquivos:

        ano = int(arquivo.parent.name.split("=")[1])
        nome = arquivo.name
        coluna_territorial = obter_coluna_territorial(ano)

        registro = {
            "ano": ano,
            "arquivo": nome,
            "coluna_territorial": coluna_territorial,
            "total_registros": None,
            "total_colunas": None,
            "ufs_encontradas": None,
            "anos_encontrados": None,
            "identificadores_nulos": None,
            "identificadores_duplicados": None,
            "status": "OK",
            "problemas": [],
        }

        try:
            # ------------------------------------------------
            # Existência e tamanho
            # ------------------------------------------------

            if not arquivo.is_file():
                raise FileNotFoundError("Arquivo inexistente.")

            if arquivo.stat().st_size == 0:
                raise ValueError("Arquivo vazio.")

            caminho = arquivo.resolve().as_posix().replace("'", "''")

            # ------------------------------------------------
            # Leitura e schema
            # ------------------------------------------------

            descricao = con.execute(
                f"""
                DESCRIBE
                SELECT * FROM read_parquet('{caminho}')
                """
            ).fetchdf()

            colunas = descricao["column_name"].tolist()
            registro["total_colunas"] = len(colunas)

            total = con.execute(
                f"""
                SELECT COUNT(*)
                FROM read_parquet('{caminho}')
                """
            ).fetchone()[0]

            registro["total_registros"] = total

            if total == 0:
                registro["problemas"].append("SEM_REGISTROS")

            # ------------------------------------------------
            # Coluna territorial histórica
            # ------------------------------------------------

            if coluna_territorial not in colunas:
                registro["problemas"].append(
                    f"COLUNA_TERRITORIAL_AUSENTE:{coluna_territorial}"
                )
            else:
                ufs = con.execute(
                    f"""
                    SELECT DISTINCT "{coluna_territorial}"
                    FROM read_parquet('{caminho}')
                    ORDER BY 1
                    """
                ).fetchall()

                ufs = [linha[0] for linha in ufs]
                registro["ufs_encontradas"] = ufs

                if ufs != [UF]:
                    registro["problemas"].append(
                        f"UF_INCORRETA:{ufs}"
                    )

            # ------------------------------------------------
            # Ano dos registros
            # ------------------------------------------------

            if "NU_ANO" in colunas:
                anos = con.execute(
                    f"""
                    SELECT DISTINCT TRY_CAST(NU_ANO AS INTEGER)
                    FROM read_parquet('{caminho}')
                    ORDER BY 1
                    """
                ).fetchall()

                anos = [linha[0] for linha in anos]
                registro["anos_encontrados"] = anos

                if anos != [ano]:
                    registro["problemas"].append(
                        f"ANO_INCORRETO:{anos}"
                    )
            else:
                registro["problemas"].append("NU_ANO_AUSENTE")

            # ------------------------------------------------
            # Identificadores nulos e duplicados
            # ------------------------------------------------

            if "NU_INSCRICAO" in colunas:
                nulos = con.execute(
                    f"""
                    SELECT COUNT(*)
                    FROM read_parquet('{caminho}')
                    WHERE NU_INSCRICAO IS NULL
                    """
                ).fetchone()[0]

                duplicados = con.execute(
                    f"""
                    SELECT COUNT(*)
                    FROM (
                        SELECT NU_INSCRICAO
                        FROM read_parquet('{caminho}')
                        WHERE NU_INSCRICAO IS NOT NULL
                        GROUP BY NU_INSCRICAO
                        HAVING COUNT(*) > 1
                    )
                    """
                ).fetchone()[0]

                registro["identificadores_nulos"] = nulos
                registro["identificadores_duplicados"] = duplicados

                if nulos > 0:
                    registro["problemas"].append(
                        f"INSCRICOES_NULAS:{nulos}"
                    )

                if duplicados > 0:
                    registro["problemas"].append(
                        f"INSCRICOES_DUPLICADAS:{duplicados}"
                    )

            # ------------------------------------------------
            # Resultado
            # ------------------------------------------------

            if registro["problemas"]:
                registro["status"] = "VERIFICAR"

        except Exception as erro:
            registro["status"] = "ERRO"
            registro["problemas"].append(
                f"{type(erro).__name__}: {erro}"
            )

        registro["problemas"] = "; ".join(registro["problemas"])
        resultados.append(registro)

finally:
    con.close()


# ============================================================
# 4. RESULTADO DETALHADO
# ============================================================

auditoria_silver = pd.DataFrame(resultados)

if not auditoria_silver.empty:
    display(
        auditoria_silver.sort_values(
            ["ano", "arquivo"]
        ).reset_index(drop=True)
    )


# ============================================================
# 5. RESUMO EXECUTIVO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO EXECUTIVO")
print("=" * 70)

print(f"Arquivos encontrados: {len(arquivos)}")
print(f"Arquivos auditados: {len(auditoria_silver)}")

if not auditoria_silver.empty:
    print("\nStatus:")
    print(
        auditoria_silver["status"]
        .value_counts(dropna=False)
        .to_string()
    )

    print("\nRegistros por ano e arquivo:")
    display(
        auditoria_silver[
            ["ano", "arquivo", "total_registros", "total_colunas", "status"]
        ]
    )

    problemas = auditoria_silver[
        auditoria_silver["status"] != "OK"
    ]

    print(f"\nArquivos com problemas: {len(problemas)}")

    if not problemas.empty:
        display(
            problemas[
                ["ano", "arquivo", "status", "problemas"]
            ]
        )

    # Esperamos 30 arquivos: 1 por ano até 2023,
    # e 2 por ano em 2024 e 2025.
    quantidade_esperada = 30
    anos_encontrados = set(auditoria_silver["ano"].tolist())

    estrutura_ok = (
        len(arquivos) == quantidade_esperada
        and anos_encontrados == ANOS_ESPERADOS
    )

    conteudo_ok = (
        (auditoria_silver["status"] == "OK").all()
        and (auditoria_silver["total_registros"] > 0).all()
    )

    print("\n" + "=" * 70)

    if estrutura_ok and conteudo_ok:
        print("RESULTADO: AUDITORIA ESTRUTURAL E BÁSICA APROVADA")
        print(
            "A Silver passou pelas verificações implementadas. "
            "Ainda é necessário comparar a cardinalidade com a origem "
            "para comprovar a preservação dos registros."
        )
    else:
        print("RESULTADO: AUDITORIA REPROVADA OU INCOMPLETA")
        print("Investigue os arquivos sinalizados antes de avançar.")

else:
    print("Nenhum arquivo foi auditado.")


AUDITORIA DA CAMADA SILVER
UF: RR
Diretório: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR
Arquivos encontrados: 30


,ano,arquivo,coluna_territorial,total_registros,total_colunas,ufs_encontradas,anos_encontrados,identificadores_nulos,identificadores_duplicados,status,problemas
0,1998,enem_1998.parquet,SG_UF_RESIDENCIA,967,27,[RR],[1998],0.0,0.0,OK,
1,1999,enem_1999.parquet,SG_UF_RESIDENCIA,1014,28,[RR],[1999],0.0,0.0,OK,
2,2000,enem_2000.parquet,SG_UF_RESIDENCIA,317,28,[RR],[2000],0.0,0.0,OK,
3,2001,enem_2001.parquet,SG_UF_ESC,2606,35,[RR],[2001],0.0,0.0,OK,
4,2002,enem_2002.parquet,SG_UF_ESC,5497,34,[RR],[2002],0.0,0.0,OK,
5,2003,enem_2003.parquet,SG_UF_ESC,6316,35,[RR],[2003],0.0,0.0,OK,
6,2004,enem_2004.parquet,SG_UF_ESC,4793,35,[RR],[2004],0.0,0.0,OK,
7,2005,enem_2005.parquet,SG_UF_ESC,5986,35,[RR],[2005],0.0,0.0,OK,
8,2006,enem_2006.parquet,SG_UF_ESC,2810,36,[RR],[2006],0.0,0.0,OK,
9,2007,enem_2007.parquet,SG_UF_ESC,4207,36,[RR],[2007],0.0,0.0,OK,



RESUMO EXECUTIVO
Arquivos encontrados: 30
Arquivos auditados: 30

Status:
status
OK    30

Registros por ano e arquivo:


,ano,arquivo,total_registros,total_colunas,status
0,1998,enem_1998.parquet,967,27,OK
1,1999,enem_1999.parquet,1014,28,OK
2,2000,enem_2000.parquet,317,28,OK
3,2001,enem_2001.parquet,2606,35,OK
4,2002,enem_2002.parquet,5497,34,OK
5,2003,enem_2003.parquet,6316,35,OK
6,2004,enem_2004.parquet,4793,35,OK
7,2005,enem_2005.parquet,5986,35,OK
8,2006,enem_2006.parquet,2810,36,OK
9,2007,enem_2007.parquet,4207,36,OK



Arquivos com problemas: 0

RESULTADO: AUDITORIA ESTRUTURAL E BÁSICA APROVADA
A Silver passou pelas verificações implementadas. Ainda é necessário comparar a cardinalidade com a origem para comprovar a preservação dos registros.


In [ ]:
#Comparação de cardinalidade entre a Bronze e a Silver

In [9]:

from pathlib import Path
import duckdb
import pandas as pd

from enem_pipeline.config import (
    SILVER_DIR,
    STAGING_DIR,
    SEPARADOR_MICRODADOS,
    obter_codificacao_microdados,
)
from enem_pipeline.bronze import (
    selecionar_csvs_dados,
    extrair_csvs_ano,
)

# ============================================================
# 1. PARÂMETROS E FUNÇÕES AUXILIARES
# ============================================================

UF = UF
ANOS = range(1998, 2026)


def coluna_territorial(ano):
    if 1998 <= ano <= 2000:
        return "SG_UF_RESIDENCIA"
    elif 2001 <= ano <= 2007:
        return "SG_UF_ESC"
    elif 2008 <= ano <= 2025:
        return "SG_UF_PROVA"

    raise ValueError(f"Ano fora do escopo: {ano}")


def nome_parquet(nome_csv, ano):
    nome = Path(nome_csv).stem.upper()

    if ano >= 2024:
        if nome == f"PARTICIPANTES_{ano}":
            return "participantes.parquet"
        if nome == f"RESULTADOS_{ano}":
            return "resultados.parquet"

        raise ValueError(f"Arquivo inesperado: {nome_csv}")

    return f"enem_{ano}.parquet"


def sql_texto(valor):
    return "'" + str(valor).replace("'", "''") + "'"


def sql_identificador(valor):
    return '"' + str(valor).replace('"', '""') + '"'


# ============================================================
# 2. COMPARAÇÃO BRONZE × SILVER
#    EXTRAÇÃO E LIMPEZA POR ANO
# ============================================================

resultados = []
erros = []

con = duckdb.connect()

try:
    for ano in ANOS:
        arquivos = selecionar_csvs_dados(ano)

        if not arquivos:
            erros.append({
                "ano": ano,
                "arquivo": None,
                "erro": "Nenhum CSV principal encontrado na Bronze.",
            })
            print(f"{ano}: nenhum CSV encontrado.")
            continue

        try:
            # Extrai somente os CSVs do ano atual.
            extrair_csvs_ano(ano)

            coluna_uf = coluna_territorial(ano)
            codificacao = obter_codificacao_microdados(ano)

            for nome_interno in arquivos:
                nome_csv = Path(nome_interno).name
                caminho_csv = (
                    STAGING_DIR / f"ano={ano}" / nome_csv
                )

                caminho_parquet = (
                    SILVER_DIR
                    / f"uf={UF}"
                    / f"ano={ano}"
                    / nome_parquet(nome_csv, ano)
                )

                if not caminho_csv.is_file():
                    raise FileNotFoundError(
                        f"CSV não encontrado: {caminho_csv}"
                    )

                if not caminho_parquet.is_file():
                    raise FileNotFoundError(
                        f"Parquet não encontrado: {caminho_parquet}"
                    )

                # Conta os registros da origem pertencentes a RR.
                total_origem = con.execute(
                    f"""
                    SELECT COUNT(*)
                    FROM read_csv(
                        {sql_texto(caminho_csv.as_posix())},
                        delim={sql_texto(SEPARADOR_MICRODADOS)},
                        header=true,
                        encoding={sql_texto(codificacao)},
                        sample_size=20000
                    )
                    WHERE {sql_identificador(coluna_uf)} = ?
                    """,
                    [UF],
                ).fetchone()[0]

                # Conta os registros já gravados na Silver.
                total_silver = con.execute(
                    f"""
                    SELECT COUNT(*)
                    FROM read_parquet(
                        {sql_texto(caminho_parquet.as_posix())}
                    )
                    """
                ).fetchone()[0]

                diferenca = total_silver - total_origem
                status = "OK" if diferenca == 0 else "DIVERGENTE"

                resultados.append({
                    "ano": ano,
                    "arquivo": nome_csv,
                    "coluna_territorial": coluna_uf,
                    "origem_filtrada": total_origem,
                    "silver": total_silver,
                    "diferenca": diferenca,
                    "status": status,
                })

                print(
                    f"{ano} | {nome_csv} | "
                    f"Origem: {total_origem:,} | "
                    f"Silver: {total_silver:,} | {status}"
                )

        except Exception as erro:
            erros.append({
                "ano": ano,
                "arquivo": None,
                "erro": f"{type(erro).__name__}: {erro}",
            })

            print(
                f"ERRO no ano {ano}: "
                f"{type(erro).__name__}: {erro}"
            )

        finally:
            # Limpa os CSVs deste ano antes de avançar.
            for nome_interno in arquivos:
                caminho_csv = (
                    STAGING_DIR
                    / f"ano={ano}"
                    / Path(nome_interno).name
                )

                try:
                    if caminho_csv.is_file():
                        caminho_csv.unlink()
                except OSError as erro_limpeza:
                    erros.append({
                        "ano": ano,
                        "arquivo": caminho_csv.name,
                        "erro": (
                            "Falha ao limpar Staging: "
                            f"{erro_limpeza}"
                        ),
                    })
                    print(
                        f"AVISO: não foi possível remover "
                        f"{caminho_csv}: {erro_limpeza}"
                    )

            print(f"Staging do ano {ano} verificado e limpo.")

finally:
    con.close()


# ============================================================
# 3. RESULTADO FINAL
# ============================================================

df_comparacao = pd.DataFrame(resultados)
df_erros = pd.DataFrame(erros)

print("\n" + "=" * 65)
print("RESULTADO DA AUDITORIA BRONZE × SILVER")
print("=" * 65)

if not df_comparacao.empty:
    display(df_comparacao)

    aprovadas = int((df_comparacao["status"] == "OK").sum())
    divergentes = int(
        (df_comparacao["status"] == "DIVERGENTE").sum()
    )

    print(f"Fontes comparadas: {len(df_comparacao)}")
    print(f"Fontes aprovadas: {aprovadas}")
    print(f"Fontes divergentes: {divergentes}")
else:
    aprovadas = 0
    divergentes = 0
    print("Nenhuma fonte foi comparada.")

print(f"Erros de processamento ou limpeza: {len(df_erros)}")

if not df_erros.empty:
    display(df_erros)

if (
    len(df_comparacao) == 30
    and aprovadas == 30
    and divergentes == 0
    and df_erros.empty
):
    print("\nAUDITORIA DE CARDINALIDADE APROVADA.")
    print("As 30 fontes preservam a quantidade de registros esperada.")
else:
    print("\nAUDITORIA INCOMPLETA OU COM DIVERGÊNCIAS.")
    print("Examine os resultados e os erros antes de avançar.")


Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=1998/MICRODADOS_ENEM_1998.csv (63,066,204 bytes)
1998 | MICRODADOS_ENEM_1998.csv | Origem: 967 | Silver: 967 | OK
Staging do ano 1998 verificado e limpo.
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=1999/MICRODADOS_ENEM_1999.csv (144,751,294 bytes)
1999 | MICRODADOS_ENEM_1999.csv | Origem: 1,014 | Silver: 1,014 | OK
Staging do ano 1999 verificado e limpo.
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2000/MICRODADOS_ENEM_2000.csv (173,638,593 bytes)
2000 | MICRODADOS_ENEM_2000.csv | Origem: 317 | Silver: 317 | OK
Staging do ano 2000 verificado e limpo.
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2001/MICRODADOS_ENEM_2001.csv (1,076,328,296 bytes)
2001 | MICRODADOS_ENEM_2001.csv | Origem: 2,606 | Silver: 2,606 | OK
Staging do ano 2001 verificado e limpo.
Extraído e conferido: /home/akel/PycharmProjects/Projet

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2005 | MICRODADOS_ENEM_2005.csv | Origem: 5,986 | Silver: 5,986 | OK
Staging do ano 2005 verificado e limpo.
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2006/MICRODADOS_ENEM_2006.csv (2,181,461,528 bytes)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006 | MICRODADOS_ENEM_2006.csv | Origem: 2,810 | Silver: 2,810 | OK
Staging do ano 2006 verificado e limpo.
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2007/MICRODADOS_ENEM_2007.csv (2,074,701,321 bytes)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (35 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025 | RESULTADOS_2025.csv | Origem: 14,158 | Silver: 14,158 | OK
Staging do ano 2025 verificado e limpo.

RESULTADO DA AUDITORIA BRONZE × SILVER


,ano,arquivo,coluna_territorial,origem_filtrada,silver,diferenca,status
0,1998,MICRODADOS_ENEM_1998.csv,SG_UF_RESIDENCIA,967,967,0,OK
1,1999,MICRODADOS_ENEM_1999.csv,SG_UF_RESIDENCIA,1014,1014,0,OK
2,2000,MICRODADOS_ENEM_2000.csv,SG_UF_RESIDENCIA,317,317,0,OK
3,2001,MICRODADOS_ENEM_2001.csv,SG_UF_ESC,2606,2606,0,OK
4,2002,MICRODADOS_ENEM_2002.csv,SG_UF_ESC,5497,5497,0,OK
5,2003,MICRODADOS_ENEM_2003.csv,SG_UF_ESC,6316,6316,0,OK
6,2004,MICRODADOS_ENEM_2004.csv,SG_UF_ESC,4793,4793,0,OK
7,2005,MICRODADOS_ENEM_2005.csv,SG_UF_ESC,5986,5986,0,OK
8,2006,MICRODADOS_ENEM_2006.csv,SG_UF_ESC,2810,2810,0,OK
9,2007,MICRODADOS_ENEM_2007.csv,SG_UF_ESC,4207,4207,0,OK


Fontes comparadas: 30
Fontes aprovadas: 30
Fontes divergentes: 0
Erros de processamento ou limpeza: 0

AUDITORIA DE CARDINALIDADE APROVADA.
As 30 fontes preservam a quantidade de registros esperada.
